## EDA — TradeMaster Pro
All 12 datasets: BTC / ETH / SOL × 1h / 4h / 12h / 1d

1. Structural integrity
2. Missing values
3. Descriptive statistics + sanity bounds
4. Infinite values
5. Target distribution — class balance
6. Multicollinearity — correlation matrix
7. Stationarity check
8. Outlier detection on returns / volume
---
**TradeMaster Pro — Platform-Specific EDA**
- TM-1. Win Rate vs Promotion Thresholds
- TM-2. Signal Quality by Session & Regime
- TM-3. Risk-to-Reward Distribution
- TM-4. Automatic Signal Risk Score
- TM-5. Multi-Asset × Multi-Timeframe Win Rate Matrix

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from statsmodels.tsa.stattools import adfuller

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 10,
})
sns.set_theme(style="whitegrid", palette="muted")


In [2]:
ASSETS     = ["BTC", "ETH", "SOL"]
TIMEFRAMES = ["1h", "4h", "12h", "1d"]
TARGET_DIR = {"1h": "target_dir_1h",  "4h": "target_dir_4h",
              "12h": "target_dir_12h", "1d": "target_dir_24h"}
TARGET_RET = {"1h": "target_ret_1h",  "4h": "target_ret_4h",
              "12h": "target_ret_12h", "1d": "target_ret_24h"}
# 3-month rolling window in bars per timeframe
ROLL_WIN   = {"1h": 2160, "4h": 540, "12h": 180, "1d": 90}

datasets = {}
for asset in ASSETS:
    for tf in TIMEFRAMES:
        key  = f"{asset}_{tf}"
        path = f"datasets/historical/{asset}_USDT_{tf}.csv"
        d    = pd.read_csv(path, parse_dates=["timestamp"]).set_index("timestamp")
        datasets[key] = d
        print(f"  {key:12s}  shape={d.shape}  {d.index.min().date()} to {d.index.max().date()}")

print(f"\nLoaded {len(datasets)} datasets.")


  BTC_1h        shape=(77684, 102)  2017-08-17 to 2026-07-03
  BTC_4h        shape=(19444, 102)  2017-08-17 to 2026-07-04
  BTC_12h       shape=(6486, 102)  2017-08-17 to 2026-07-04
  BTC_1d        shape=(3244, 102)  2017-08-17 to 2026-07-04
  ETH_1h        shape=(77684, 102)  2017-08-17 to 2026-07-03
  ETH_4h        shape=(19444, 102)  2017-08-17 to 2026-07-04
  ETH_12h       shape=(6486, 102)  2017-08-17 to 2026-07-04
  ETH_1d        shape=(3244, 102)  2017-08-17 to 2026-07-04
  SOL_1h        shape=(50892, 102)  2020-09-11 to 2026-07-03
  SOL_4h        shape=(12735, 102)  2020-09-11 to 2026-07-04
  SOL_12h       shape=(4245, 102)  2020-09-11 to 2026-07-04
  SOL_1d        shape=(2123, 102)  2020-09-11 to 2026-07-04

Loaded 12 datasets.


### 1. Structural Integrity

In [3]:
FREQ_MAP = {"1h": "h", "4h": "4h", "12h": "12h", "1d": "D"}

rows = []
for asset in ASSETS:
    for tf in TIMEFRAMES:
        key = f"{asset}_{tf}"
        d   = datasets[key]
        expected = pd.date_range(d.index.min(), d.index.max(), freq=FREQ_MAP[tf])
        missing  = expected.difference(d.index)
        rows.append({
            "Dataset":      key,
            "Rows":         len(d),
            "Cols":         d.shape[1],
            "Start":        str(d.index.min().date()),
            "End":          str(d.index.max().date()),
            "Missing bars": len(missing),
            "Dup index":    int(d.index.duplicated().sum()),
        })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))


Dataset  Rows  Cols      Start        End  Missing bars  Dup index
 BTC_1h 77684   102 2017-08-17 2026-07-03           128          0
 BTC_4h 19444   102 2017-08-17 2026-07-04            16          0
BTC_12h  6486   102 2017-08-17 2026-07-04             1          0
 BTC_1d  3244   102 2017-08-17 2026-07-04             0          0
 ETH_1h 77684   102 2017-08-17 2026-07-03           128          0
 ETH_4h 19444   102 2017-08-17 2026-07-04            16          0
ETH_12h  6486   102 2017-08-17 2026-07-04             1          0
 ETH_1d  3244   102 2017-08-17 2026-07-04             0          0
 SOL_1h 50892   102 2020-09-11 2026-07-03            20          0
 SOL_4h 12735   102 2020-09-11 2026-07-04             0          0
SOL_12h  4245   102 2020-09-11 2026-07-04             0          0
 SOL_1d  2123   102 2020-09-11 2026-07-04             0          0


### 2. Missing Values

In [ ]:
nan_totals = {}
for key, d in datasets.items():
    nan_totals[key] = d.isna().sum()

nan_df = pd.DataFrame(nan_totals).T
nan_df = nan_df.loc[:, (nan_df > 0).any()]

print(f"Features with NaN in at least one dataset: {nan_df.shape[1]}")
print("\nNaN count per dataset (non-zero columns only):")
print(nan_df.to_string())

# Heatmap — clip at 200 for colour clarity
fig, ax = plt.subplots(figsize=(20, 6))
sns.heatmap(
    nan_df.clip(upper=200), cmap="YlOrRd", ax=ax,
    linewidths=0.3, cbar_kws={"label": "NaN count (capped 200)"},
    xticklabels=True, yticklabels=True
)
ax.set_title("Missing Values Heatmap — All 12 Datasets", fontsize=13, fontweight="bold")
ax.tick_params(axis="x", rotation=90, labelsize=6)
ax.tick_params(axis="y", labelsize=8)
plt.tight_layout()
plt.savefig("eda_missing_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: eda_missing_heatmap.png")


### 3. Descriptive Statistics + Sanity Bounds

In [5]:
rows = []
for key, d in datasets.items():
    # RSI_14 bounds [0, 100]
    rsi = d["rsi_14"].dropna()
    rsi_invalid = int((~rsi.between(0, 100)).sum())

    # BB_pct bounds [-5, 5]
    bb = d["bb_pct"].dropna() if "bb_pct" in d.columns else pd.Series(dtype=float)
    bb_invalid = int((~bb.between(-5, 5)).sum()) if len(bb) else 0

    # ATR must be > 0
    atr = d["atr_14"].dropna()
    atr_invalid = int((atr <= 0).sum())

    rows.append({
        "Dataset":       key,
        "RSI invalid":   rsi_invalid,
        "BB_pct invalid":bb_invalid,
        "ATR<=0":        atr_invalid,
        "close min":     round(d["close"].min(), 2),
        "close max":     round(d["close"].max(), 2),
        "ret_1 min":     round(d["ret_1"].min(), 4) if "ret_1" in d.columns else None,
        "ret_1 max":     round(d["ret_1"].max(), 4) if "ret_1" in d.columns else None,
    })

print(pd.DataFrame(rows).to_string(index=False))


Dataset  RSI invalid  BB_pct invalid  ATR<=0  close min  close max  ret_1 min  ret_1 max
 BTC_1h            0               0       0    2919.00  126011.18    -0.2010     0.1603
 BTC_4h            0               0       0    2919.00  125410.81    -0.2294     0.2716
BTC_12h            0               0       0    2919.00  124658.54    -0.2684     0.2371
 BTC_1d            0               0       0    3189.02  124658.54    -0.5026     0.2030
 ETH_1h            0               0       0      82.17    4935.00    -0.2341     0.1662
 ETH_4h            0               0       0      82.17    4846.71    -0.2495     0.2747
ETH_12h            0               0       0      83.67    4832.07    -0.3540     0.2387
 ETH_1d            0               0       0      83.76    4832.07    -0.5905     0.2338
 SOL_1h            0               0       0       1.18     286.24    -0.2142     0.1710
 SOL_4h            0               0       0       1.20     286.24    -0.2213     0.2686
SOL_12h            0 

### 4. Infinite Values

In [6]:
rows = []
for key, d in datasets.items():
    num    = d.select_dtypes(include=[np.number])
    n_inf  = int(np.isinf(num).sum().sum())
    rows.append({"Dataset": key, "Inf count": n_inf,
                 "Status": "Clean" if n_inf == 0 else "Has Inf"})

inf_df = pd.DataFrame(rows)
print(inf_df.to_string(index=False))
print(f"\nDatasets with Inf: {(inf_df['Inf count'] > 0).sum()}")


Dataset  Inf count Status
 BTC_1h          0  Clean
 BTC_4h          0  Clean
BTC_12h          0  Clean
 BTC_1d          0  Clean
 ETH_1h          0  Clean
 ETH_4h          0  Clean
ETH_12h          0  Clean
 ETH_1d          0  Clean
 SOL_1h          0  Clean
 SOL_4h          0  Clean
SOL_12h          0  Clean
 SOL_1d          0  Clean

Datasets with Inf: 0


### 5. Target Distribution — Class Balance

In [ ]:
label_map   = {0.0: "Down", 1.0: "Flat", 2.0: "Up"}
class_colors = {"Down": "#e74c3c", "Flat": "#95a5a6", "Up": "#2ecc71"}

# Build a tidy DataFrame: dataset × class → pct
rows = []
for asset in ASSETS:
    for tf in TIMEFRAMES:
        key = f"{asset}_{tf}"
        vc  = datasets[key][TARGET_DIR[tf]].value_counts(normalize=True, dropna=True).sort_index()
        for cls, pct in vc.items():
            rows.append({"Dataset": key, "Class": label_map.get(cls, str(cls)), "Pct": pct * 100})

dist_df = pd.DataFrame(rows)

# --- Stacked horizontal bar (one bar per dataset) ---
pivot   = dist_df.pivot(index="Dataset", columns="Class", values="Pct")[["Down", "Flat", "Up"]]
order   = [f"{a}_{tf}" for a in ASSETS for tf in TIMEFRAMES]
pivot   = pivot.reindex(order)

fig, ax = plt.subplots(figsize=(10, 8))
left = np.zeros(len(pivot))
for cls in ["Down", "Flat", "Up"]:
    vals = pivot[cls].values
    bars = ax.barh(pivot.index, vals, left=left, color=class_colors[cls],
                   label=cls, edgecolor="white", linewidth=0.4)
    # annotate if wide enough
    for i, (v, l) in enumerate(zip(vals, left)):
        if v > 5:
            ax.text(l + v / 2, i, f"{v:.0f}%", ha="center", va="center",
                    fontsize=7, color="white", fontweight="bold")
    left += vals

ax.axvline(33.3, color="black", linestyle="--", linewidth=0.9, alpha=0.6, label="Equal split")
ax.set_xlim(0, 100)
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_xlabel("Share of bars")
ax.set_title("Target Direction Class Balance — All 12 Datasets", fontsize=13, fontweight="bold")
ax.legend(loc="lower right", fontsize=9)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("eda_target_dist.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: eda_target_dist.png")

# Summary table
print(pivot.round(1).to_string())


### 6. Multicollinearity — Correlation Matrix
One heatmap per asset (using 1h data). Features are identical across timeframes; 1h gives maximum sample size.

In [ ]:
feature_groups = {
    "Price/Returns":   ["ret_1","ret_3","ret_6","ret_12","ret_24","ret_48","gap_pct"],
    "Momentum":        ["rsi_7","rsi_14","rsi_21","macd","macd_signal","macd_hist",
                        "stoch_k","stoch_d","willr","cci","momentum"],
    "Trend":           ["adx","plus_di","minus_di","ema50_slope","ema200_slope",
                        "trend_strength","trend_consistency",
                        "ema_cross_9_21","ema_cross_21_50","ema_cross_50_200"],
    "Volatility":      ["atr_14","atr_21","atr_ratio","bb_width","bb_pct",
                        "vol_ratio","vol_stddev","vol_regime_20","vol_regime_ratio","is_high_vol"],
    "Ichimoku":        ["tenkan","kijun","senkou_a","senkou_b","cloud_bull","tenkan_kijun_cross"],
    "S/R & Price Pos": ["pct_from_high","pct_from_low","dist_resist","dist_support","sr_width",
                        "vs_ema_9","vs_ema_21","vs_ema_50","vs_ema_100","vs_ema_200","vs_vwap",
                        "dd_from_ath"],
}

for asset in ASSETS:
    d = datasets[f"{asset}_1h"]
    fig, axes = plt.subplots(2, 3, figsize=(22, 14))
    axes = axes.flatten()
    for ax, (grp, cols) in zip(axes, feature_groups.items()):
        valid = [c for c in cols if c in d.columns]
        corr  = d[valid].corr()
        mask  = np.triu(np.ones_like(corr, dtype=bool))
        sns.heatmap(
            corr, mask=mask, ax=ax, cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, annot=len(valid) <= 8, fmt=".2f",
            linewidths=0.3, xticklabels=True, yticklabels=True,
            square=True, cbar_kws={"shrink": 0.8}
        )
        ax.set_title(grp, fontsize=11, fontweight="bold")
        ax.tick_params(axis="x", rotation=45, labelsize=7)
        ax.tick_params(axis="y", labelsize=7)
    plt.suptitle(f"{asset} 1h — Feature Correlation Heatmaps", fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    fname = f"eda_corr_{asset}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {fname}")

# Global summary heatmap — one heatmap across all feature groups, all features, BTC 1h
all_feats = [c for g in feature_groups.values() for c in g]
d = datasets["BTC_1h"]
feats = [c for c in all_feats if c in d.columns]
corr_all = d[feats].corr()
mask_all  = np.triu(np.ones_like(corr_all, dtype=bool))
fig, ax  = plt.subplots(figsize=(20, 18))
sns.heatmap(
    corr_all, mask=mask_all, ax=ax, cmap="RdBu_r", center=0,
    vmin=-1, vmax=1, linewidths=0.2, xticklabels=True, yticklabels=True,
    cbar_kws={"shrink": 0.6, "label": "Pearson r"}
)
ax.set_title("BTC 1h — Full Feature Correlation Matrix", fontsize=14, fontweight="bold")
ax.tick_params(axis="x", rotation=90, labelsize=6)
ax.tick_params(axis="y", labelsize=6)
plt.tight_layout()
plt.savefig("eda_corr_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: eda_corr_heatmap.png")

# High-corr pairs summary
print("\n=== Highly correlated pairs (|r| > 0.90) per asset ===")
for asset in ASSETS:
    d     = datasets[f"{asset}_1h"]
    valid = [c for c in all_feats if c in d.columns]
    corr  = d[valid].corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    pairs = upper.stack().reset_index()
    pairs.columns = ["A", "B", "r"]
    pairs = pairs[pairs["r"] > 0.90].sort_values("r", ascending=False)
    print(f"  {asset}: {len(pairs)} pairs")
    if len(pairs):
        print(pairs.head(10).to_string(index=False))


### 7. Stationarity Check
ADF test on key features for all 12 datasets (subsampled to 5 000 rows for speed).

In [9]:
test_feats = ["close","ret_1","rsi_14","macd","bb_pct","atr_14","vs_ema_50","vol_ratio","adx"]

rows = []
for key, d in datasets.items():
    for feat in test_feats:
        if feat not in d.columns:
            continue
        series = d[feat].dropna()
        series = series.iloc[::max(1, len(series)//5000)]   # subsample
        stat, p = adfuller(series, autolag="AIC")[:2]
        rows.append({"Dataset": key, "Feature": feat,
                     "ADF": round(stat, 3), "p": round(p, 4),
                     "Stationary": "Yes" if p < 0.05 else "No"})

stat_df = pd.DataFrame(rows)

# Pivot: rows=features, cols=datasets, values=Yes/No
pivot = stat_df.pivot(index="Feature", columns="Dataset", values="Stationary")
print(pivot.to_string())

non_stat = stat_df[stat_df["Stationary"] == "No"]
print(f"\nNon-stationary feature-dataset pairs: {len(non_stat)}")
if len(non_stat):
    print(non_stat[["Dataset","Feature","p"]].to_string(index=False))


Dataset   BTC_12h BTC_1d BTC_1h BTC_4h ETH_12h ETH_1d ETH_1h ETH_4h SOL_12h SOL_1d SOL_1h SOL_4h
Feature                                                                                         
adx           Yes    Yes    Yes    Yes     Yes    Yes    Yes    Yes     Yes    Yes    Yes    Yes
atr_14        Yes     No    Yes    Yes     Yes    Yes    Yes    Yes     Yes     No    Yes    Yes
bb_pct        Yes    Yes    Yes    Yes     Yes    Yes    Yes    Yes     Yes    Yes    Yes    Yes
close          No     No     No     No      No     No     No     No      No     No     No     No
macd          Yes    Yes    Yes    Yes     Yes    Yes    Yes    Yes     Yes    Yes    Yes    Yes
ret_1         Yes    Yes    Yes    Yes     Yes    Yes    Yes    Yes     Yes    Yes    Yes    Yes
rsi_14        Yes    Yes    Yes    Yes     Yes    Yes    Yes    Yes     Yes    Yes    Yes    Yes
vol_ratio     Yes    Yes    Yes    Yes     Yes    Yes    Yes    Yes     Yes    Yes    Yes    Yes
vs_ema_50     Yes    Yes    Ye

### 8. Outlier Detection on Returns & Volume

In [ ]:
check_cols = ["ret_1", "ret_3", "ret_6", "volume", "vol_ratio", "atr_14"]

# Summary table
rows = []
for key, d in datasets.items():
    row = {"Dataset": key}
    for col in check_cols:
        if col not in d.columns:
            row[col] = None; continue
        s = d[col].dropna()
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr     = q3 - q1
        n_out   = int(((s < q1 - 3*iqr) | (s > q3 + 3*iqr)).sum())
        row[col] = f"{n_out/len(s)*100:.2f}%"
    rows.append(row)

out_df = pd.DataFrame(rows)
print("Outlier rate (3×IQR) per dataset:")
print(out_df.to_string(index=False))

# --- Box plot grid: ret_1 per dataset, grouped by asset ---
# Collect data as list-of-series keyed by dataset label
asset_colors = {"BTC": "#f39c12", "ETH": "#3498db", "SOL": "#9b59b6"}

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=False)
for ax, asset in zip(axes, ASSETS):
    data   = []
    labels = []
    for tf in TIMEFRAMES:
        key = f"{asset}_{tf}"
        s   = datasets[key]["ret_1"].dropna()
        data.append(s.values)
        labels.append(tf)

    bp = ax.boxplot(
        data, labels=labels, patch_artist=True, notch=False,
        medianprops=dict(color="black", linewidth=2),
        whiskerprops=dict(linewidth=1.2),
        capprops=dict(linewidth=1.2),
        flierprops=dict(marker=".", markersize=1.5, alpha=0.3,
                        color=asset_colors[asset]),
        whis=3.0,          # 3×IQR whiskers — matches outlier definition
    )
    for patch in bp["boxes"]:
        patch.set_facecolor(asset_colors[asset])
        patch.set_alpha(0.6)

    ax.set_title(f"{asset} — ret_1 Distribution", fontsize=11, fontweight="bold")
    ax.set_xlabel("Timeframe")
    ax.set_ylabel("1-bar return" if asset == "BTC" else "")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
    ax.axhline(0, color="grey", linewidth=0.8, linestyle="--")

plt.suptitle("Return Distribution with 3×IQR Whiskers — Outlier dots beyond fences",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("eda_outliers.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: eda_outliers.png")


---
## TradeMaster Pro — Platform-Specific EDA
Directly maps dataset features to signal quality, win-rate thresholds, and risk scoring.

### TM-1. Win Rate vs Promotion Thresholds
Rolling 3-month win rate for all 12 datasets plotted against the platform's 60% / 70% / 75% thresholds.

In [ ]:
THRESH = {"Bronze": 60, "Silver": 70, "Gold": 75}
THRESH_COLORS = {"Bronze": "#cd7f32", "Silver": "#aaa9ad", "Gold": "#ffd700"}

fig, axes = plt.subplots(3, 4, figsize=(22, 13))
summary_rows = []

for r, asset in enumerate(ASSETS):
    for c, tf in enumerate(TIMEFRAMES):
        ax  = axes[r][c]
        key = f"{asset}_{tf}"
        d   = datasets[key]
        tc  = TARGET_DIR[tf]
        sub = d[d[tc].notna()].copy()
        sub["tp1_hit"] = (sub[tc] == 2).astype(int)
        wr_overall = sub["tp1_hit"].mean()
        rolling    = sub["tp1_hit"].rolling(ROLL_WIN[tf], min_periods=ROLL_WIN[tf]//4).mean() * 100

        ax.plot(sub.index, rolling, color="royalblue", linewidth=0.8, alpha=0.9, zorder=3)

        # Shaded bands between thresholds
        ax.fill_between(sub.index, 60, 70,
                        alpha=0.08, color=THRESH_COLORS["Bronze"], zorder=1)
        ax.fill_between(sub.index, 70, 75,
                        alpha=0.10, color=THRESH_COLORS["Silver"], zorder=1)
        ax.fill_between(sub.index, 75, 100,
                        alpha=0.10, color=THRESH_COLORS["Gold"],   zorder=1)

        for name, val in THRESH.items():
            ax.axhline(val, color=THRESH_COLORS[name], linestyle="--",
                       linewidth=1.1, label=f"{name} {val}%", zorder=2)

        ax.set_ylim(10, 90)
        ax.set_title(f"{key}  (μ={wr_overall*100:.1f}%)", fontsize=8, fontweight="bold")
        ax.tick_params(labelsize=6)
        if r == 0 and c == 0:
            ax.legend(fontsize=6, loc="upper left")

        above60 = (rolling >= 60).mean()
        above75 = (rolling >= 75).mean()
        summary_rows.append({"Dataset": key, "Overall WR%": round(wr_overall*100, 2),
                              "% time >60%": round(above60*100, 1),
                              "% time >75%": round(above75*100, 1)})

plt.suptitle("Rolling 3-Month Win Rate vs TradeMaster Promotion Thresholds",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("tm_win_rate.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: tm_win_rate.png\n")
print(pd.DataFrame(summary_rows).to_string(index=False))


### TM-2. Signal Quality by Session & Volatility Regime
Win rate breakdown by Asia / EU / US session and High / Low vol for all 12 datasets.

In [ ]:
rows = []
for asset in ASSETS:
    for tf in TIMEFRAMES:
        key = f"{asset}_{tf}"
        d   = datasets[key]
        tc  = TARGET_DIR[tf]
        sub = d[d[tc].notna()].copy()
        sub["tp1_hit"] = (sub[tc] == 2).astype(int)

        for sess, col in [("Asia","session_asia"),("EU","session_eu"),("US","session_us")]:
            if col not in sub.columns: continue
            grp = sub[sub[col]==1]["tp1_hit"]
            rows.append({"Dataset": key, "Segment": sess,
                         "WR%": round(grp.mean()*100, 2), "n": len(grp)})

        for regime, label in [(0,"Low Vol"),(1,"High Vol")]:
            grp = sub[sub["is_high_vol"]==regime]["tp1_hit"]
            rows.append({"Dataset": key, "Segment": label,
                         "WR%": round(grp.mean()*100, 2), "n": len(grp)})

sess_df = pd.DataFrame(rows)

# Pivot: rows=Dataset, cols=Segment
segment_order = ["Asia", "EU", "US", "Low Vol", "High Vol"]
pivot = sess_df.pivot(index="Dataset", columns="Segment", values="WR%")[segment_order]
dataset_order = [f"{a}_{tf}" for a in ASSETS for tf in TIMEFRAMES]
pivot = pivot.reindex(dataset_order)

# Heatmap — center at unconditional mean (~33%), not 50%
fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(
    pivot, annot=True, fmt=".1f", cmap="RdYlGn", center=33,
    vmin=0, vmax=50, linewidths=0.5, ax=ax,
    cbar_kws={"label": "Win Rate (%)", "shrink": 0.8}
)
ax.set_title("Win Rate (%) by Session & Vol Regime — All 12 Datasets\n"
             "(green = above 33% base rate)", fontsize=12, fontweight="bold")
ax.set_xlabel(""); ax.set_ylabel("")
plt.tight_layout()
plt.savefig("tm_session_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: tm_session_heatmap.png\n")
print(pivot.to_string())

# --- Hour × Day-of-Week heatmap — BTC 1h reference ---
d1h = datasets["BTC_1h"]
sub = d1h[d1h[TARGET_DIR["1h"]].notna()].copy()
sub["tp1_hit"] = (sub[TARGET_DIR["1h"]] == 2).astype(int)
days = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
pvt  = sub.pivot_table(values="tp1_hit", index="hour", columns="day_of_week", aggfunc="mean")
pvt.columns = days

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(
    pvt * 100, cmap="RdYlGn", center=pvt.mean().mean()*100,
    vmin=0, ax=ax, annot=True, fmt=".0f",
    linewidths=0.3, cbar_kws={"label": "Win Rate (%)"}
)
ax.set_title("BTC 1h — Win Rate % by Hour × Day of Week", fontsize=12, fontweight="bold")
ax.set_xlabel("Day of Week"); ax.set_ylabel("Hour (UTC)")
plt.tight_layout()
plt.savefig("tm_hour_dow_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: tm_hour_dow_heatmap.png")


### TM-3. Risk-to-Reward (R:R) Distribution
ATR-based SL proxy vs actual forward return for all 12 datasets.

In [ ]:
rr_rows = []
for asset in ASSETS:
    for tf in TIMEFRAMES:
        key = f"{asset}_{tf}"
        d   = datasets[key]
        tr  = TARGET_RET[tf]
        sub = d[["atr_14","close",tr,"is_high_vol"]].dropna().copy()
        sub["rr"] = (sub[tr] * sub["close"]).abs() / sub["atr_14"].replace(0, np.nan)
        sub = sub[sub["rr"].between(0, 10)]

        med = sub["rr"].median()
        mn  = sub["rr"].mean()
        pct_above15 = (sub["rr"] > 1.5).mean() * 100
        rr_rows.append({"Dataset": key, "Median R:R": round(med, 3),
                        "Mean R:R": round(mn, 3), ">1.5 R:R %": round(pct_above15, 1),
                        "_sub": sub})

# --- Overlaid KDE split by vol regime, 3×4 grid ---
fig, axes = plt.subplots(3, 4, figsize=(20, 12))

for idx, (r, asset) in enumerate([(r, a) for r, a in enumerate(ASSETS)]):
    for c, tf in enumerate(TIMEFRAMES):
        ax   = axes[r][c]
        row  = rr_rows[r * 4 + c]
        sub  = row["_sub"]
        med  = row["Median R:R"]
        p15  = row[">1.5 R:R %"]

        for vol_val, label, color in [(0,"Low Vol","#3498db"),(1,"High Vol","#e74c3c")]:
            grp = sub[sub["is_high_vol"]==vol_val]["rr"]
            if len(grp) > 10:
                grp.plot.hist(ax=ax, bins=60, density=True, alpha=0.45,
                              color=color, label=label, edgecolor="none")

        ax.axvline(1.0, color="black",  linestyle="--", linewidth=1,   label="R:R=1")
        ax.axvline(med, color="green",  linestyle="-",  linewidth=1.4, label=f"med={med:.2f}")
        ax.set_xlim(0, 6)
        ax.set_title(f"{asset}_{tf}  ({p15:.1f}% > 1.5×)", fontsize=8, fontweight="bold")
        ax.set_xlabel("R:R", fontsize=7)
        ax.tick_params(labelsize=6)
        if r == 0 and c == 0:
            ax.legend(fontsize=6)

plt.suptitle("Risk-to-Reward Distribution — Low Vol vs High Vol Regime\n(ATR-based SL proxy)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("tm_rr_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: tm_rr_distribution.png\n")

summary_df = pd.DataFrame([{k: v for k, v in r.items() if k != "_sub"} for r in rr_rows])
print(summary_df.to_string(index=False))


### TM-4. Automatic Signal Risk Score
LOW / MEDIUM / HIGH per bar across all 12 datasets, with win rate per tier.

In [ ]:
risk_rows = []
for asset in ASSETS:
    for tf in TIMEFRAMES:
        key = f"{asset}_{tf}"
        d   = datasets[key]
        tc  = TARGET_DIR[tf]
        sub = d[["atr_ratio","dd_from_ath","is_high_vol",tc]].dropna().copy()

        conds = [
            (sub["is_high_vol"]==1) & (sub["atr_ratio"]>1.5) & (sub["dd_from_ath"]<-0.30),
            (sub["is_high_vol"]==0) & (sub["atr_ratio"]<1.0) & (sub["dd_from_ath"]>-0.15),
        ]
        sub["risk"] = np.select(conds, ["HIGH","LOW"], default="MEDIUM")
        sub["tp1"]  = (sub[tc] == 2).astype(int)

        dist = sub["risk"].value_counts(normalize=True) * 100
        wr   = sub.groupby("risk")["tp1"].mean() * 100

        risk_rows.append({
            "Dataset": key,
            "LOW %":   round(dist.get("LOW",   0), 1),
            "MED %":   round(dist.get("MEDIUM",0), 1),
            "HIGH %":  round(dist.get("HIGH",  0), 1),
            "WR LOW":  round(wr.get("LOW",  float("nan")), 1),
            "WR MED":  round(wr.get("MEDIUM",float("nan")),1),
            "WR HIGH": round(wr.get("HIGH", float("nan")), 1),
        })

risk_df = pd.DataFrame(risk_rows)
print(risk_df.to_string(index=False))

tier_colors = {"LOW": "#2ecc71", "MED": "#f39c12", "HIGH": "#e74c3c"}

fig, (ax_top, ax_bot) = plt.subplots(
    2, 1, figsize=(18, 10),
    gridspec_kw={"height_ratios": [1, 1.4]}
)
x = np.arange(len(risk_df))

# --- Panel 1: Stacked bar — tier composition ---
left = np.zeros(len(risk_df))
for tier, col, color in [("LOW","LOW %",tier_colors["LOW"]),
                          ("MED","MED %",tier_colors["MED"]),
                          ("HIGH","HIGH %",tier_colors["HIGH"])]:
    vals = risk_df[col].fillna(0).values
    ax_top.bar(x, vals, bottom=left, color=color, label=tier,
               edgecolor="white", linewidth=0.4)
    for i, (v, l) in enumerate(zip(vals, left)):
        if v > 3:
            ax_top.text(i, l + v/2, f"{v:.0f}%", ha="center", va="center",
                        fontsize=6.5, color="white", fontweight="bold")
    left += vals

ax_top.set_xticks(x)
ax_top.set_xticklabels(risk_df["Dataset"], rotation=45, ha="right", fontsize=8)
ax_top.set_ylabel("Share of bars (%)")
ax_top.set_title("Risk Tier Distribution — All 12 Datasets", fontsize=11, fontweight="bold")
ax_top.legend(loc="upper right", fontsize=9)
ax_top.set_ylim(0, 110)

# --- Panel 2: Dot / lollipop — win rate per tier ---
w = 0.28
for i, (tier, col, color) in enumerate([("LOW","WR LOW",tier_colors["LOW"]),
                                          ("MED","WR MED",tier_colors["MED"])]):
    wr = risk_df[col].values
    offset = (i - 0.5) * w
    for xi, yi in zip(x + offset, wr):
        if not np.isnan(yi):
            ax_bot.plot([xi, xi], [0, yi], color=color, linewidth=1.4, alpha=0.6)
            ax_bot.scatter(xi, yi, color=color, s=50, zorder=5, label=tier if xi == x[0] + offset else "")

ax_bot.axhline(60, color="grey", linestyle="--", linewidth=1.2, label="60% target")
ax_bot.set_xticks(x)
ax_bot.set_xticklabels(risk_df["Dataset"], rotation=45, ha="right", fontsize=8)
ax_bot.set_ylabel("Win Rate (%)")
ax_bot.set_title("Win Rate by Risk Tier — Lollipop Chart\n(HIGH tier never triggers — thresholds need recalibration)",
                 fontsize=11, fontweight="bold")
ax_bot.legend(fontsize=9, loc="upper left")
ax_bot.set_ylim(0, 70)

plt.tight_layout()
plt.savefig("tm_risk_score.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: tm_risk_score.png")


### TM-5. Multi-Asset × Multi-Timeframe Win Rate Matrix

In [ ]:
results = []
for asset in ASSETS:
    for tf in TIMEFRAMES:
        key   = f"{asset}_{tf}"
        d     = datasets[key]
        tc    = TARGET_DIR[tf]
        valid = d[d[tc].notna()]
        wr    = (valid[tc] == 2).mean()
        results.append({"Asset": asset, "Timeframe": tf,
                        "Win Rate": round(wr*100, 2), "Bars": len(valid)})

wr_df    = pd.DataFrame(results)
pivot_wr = wr_df.pivot(index="Timeframe", columns="Asset", values="Win Rate").reindex(["1h","4h","12h","1d"])

# Heatmap — center at 33% (unconditional base rate), not 50%
fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(
    pivot_wr, annot=True, fmt=".1f", cmap="RdYlGn",
    center=33, vmin=10, vmax=50,
    linewidths=0.5, ax=ax,
    cbar_kws={"label": "Win Rate (%)", "shrink": 0.8}
)
# Annotate each cell with bar count
for i, tf in enumerate(pivot_wr.index):
    for j, asset in enumerate(pivot_wr.columns):
        bars = wr_df[(wr_df["Asset"]==asset) & (wr_df["Timeframe"]==tf)]["Bars"].values[0]
        ax.text(j + 0.5, i + 0.78, f"n={bars:,}", ha="center", va="center",
                fontsize=6.5, color="dimgrey")

ax.set_title("Unconditional BUY Win Rate (%) — All Assets × Timeframes\n"
             "(base rate ≈ 33%  |  n = sample size per cell)",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Asset"); ax.set_ylabel("Timeframe")
plt.tight_layout()
plt.savefig("tm_asset_tf_winrate.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: tm_asset_tf_winrate.png")
print("\nFull table:")
print(wr_df.to_string(index=False))
